# Credit Risk Scorecard Development
### Give Me Some Credit — Application Scoring Model
**Author:** Denzel C. Seals | Senior Data Analyst  
**Dataset:** [Give Me Some Credit](https://www.kaggle.com/c/GiveMeSomeCredit) — 150,000 loan applicants  
**Stack:** Python · Scikit-learn · Pandas · Plotly · Streamlit

---

## Executive Summary

This notebook develops an industry-standard **Weight of Evidence (WOE) scorecard** for predicting
the probability that an applicant will experience serious delinquency (90+ DPD) within 2 years.

The scorecard follows the PDO (Points to Double Odds) methodology used by major consumer lenders:
- **Base score:** 600
- **PDO:** 20 (score increases by 20 for every doubling of good:bad odds)
- **Score range:** approximately 400–800

**Model validation results (held-out test set):**

| Metric | Value | Interpretation |
|---|---|---|
| AUC-ROC | ~0.86 | Strong discrimination |
| Gini | ~0.72 | Industry-competitive |
| KS Statistic | ~0.38 | Good separation |

*Actual values depend on your data — results will appear after running all cells.*

---

## Workflow

```
Raw Data → Clean & Split → WOE Binning → Feature Selection (IV)
→ Logistic Regression → Points Scaling → Validation → Cut-off Analysis → Save Artifacts
```


## 1. Setup

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

from src.preprocessing import load_and_clean, split_data, feature_summary
from src.woe_binning import WOEBinner
from src.scorecard import CreditScorecard
from src.validation import (
    full_validation_report, roc_curve_data, ks_curve_data,
    score_distribution, compute_ks, compute_gini, compute_auc,
)
from src.visualization import (
    plot_iv_ranking, plot_woe_bins, plot_score_distribution,
    plot_roc_curve, plot_ks_curve, plot_cutoff_analysis,
)

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
print("Setup complete.")


Setup complete.


## 2. Data Loading & Exploratory Analysis

The Give Me Some Credit dataset contains **150,000 consumer loan applicants** with
10 behavioral and demographic features. The target variable is `SeriousDlqin2yrs`
— whether the applicant experienced 90+ DPD within 2 years of observation.

**Key characteristics:**
- Highly imbalanced: ~6.7% bad rate (industry-realistic for prime/near-prime portfolios)
- Two features have significant missing rates: `MonthlyIncome` (~20%) and `NumberOfDependents` (~2.6%)
- Several features have extreme outliers requiring capping


In [2]:
df = load_and_clean("../data/raw/cs-training.csv")
df.head()


Loaded: 150,000 rows, 11 columns
Duplicates removed: 609

Missing values:
  monthly_income: 29,221 (19.6%)
  dependents: 3,828 (2.6%)

Final dataset: 149,391 rows
Bad rate: 6.70%


,bad_flag,revolving_util,age,dpd_30_59_cnt,debt_ratio,monthly_income,open_credit_lines,dpd_90_cnt,real_estate_loans,dpd_60_89_cnt,dependents
1,1,0.7661,45,2,0.8030,9120.0000,13,0,6,0,2.0000
2,0,0.9572,40,0,0.1219,2600.0000,4,0,0,0,1.0000
3,0,0.6582,38,1,0.0851,3042.0000,2,1,0,0,0.0000
4,0,0.2338,30,0,0.0360,3300.0000,5,0,0,0,0.0000
5,0,0.9072,49,1,0.0249,63588.0000,7,0,1,0,0.0000


In [3]:
# Feature summary: missingness, range, correlation with target
feature_summary(df)


,dtype,missing,pct_miss,mean,min,max,bad_rate_corr
feature,,,,,,,
revolving_util,float64,0,0.0%,0.3185,0.0000,1.0000,0.2793
age,int64,0,0.0%,52.3062,0.0000,109.0000,-0.1157
dpd_30_59_cnt,int64,0,0.0%,0.2614,0.0000,10.0000,0.2796
debt_ratio,float64,0,0.0%,1.2920,0.0000,5.0000,-0.0141
monthly_income,float64,29221,19.6%,6675.0983,0.0000,3008750.0000,-0.0136
open_credit_lines,int64,0,0.0%,8.4780,0.0000,40.0000,-0.0308
dpd_90_cnt,int64,0,0.0%,0.1053,0.0000,10.0000,0.2990
real_estate_loans,int64,0,0.0%,1.0193,0.0000,10.0000,-0.0097
dpd_60_89_cnt,int64,0,0.0%,0.0800,0.0000,10.0000,0.2368


In [4]:
# Target distribution
print(f"Bad (1): {df['bad_flag'].sum():,} ({df['bad_flag'].mean()*100:.2f}%)")
print(f"Good (0): {(1-df['bad_flag']).sum():,} ({(1-df['bad_flag']).mean()*100:.2f}%)")


Bad (1): 10,009 (6.70%)
Good (0): 139,382 (93.30%)


## 3. Train / Test Split

Stratified 70/30 split on the target variable to preserve bad rate in both sets.
All WOE bins and model coefficients are fit **exclusively on the training set**
to prevent data leakage into validation.


In [5]:
X_train, X_test, y_train, y_test = split_data(df)
print(f"\nFeatures: {list(X_train.columns)}")


Train: 104,573 rows | Bad rate: 6.70%
Test:  44,818 rows  | Bad rate: 6.70%

Features: ['revolving_util', 'age', 'dpd_30_59_cnt', 'debt_ratio', 'monthly_income', 'open_credit_lines', 'dpd_90_cnt', 'real_estate_loans', 'dpd_60_89_cnt', 'dependents']


## 4. Weight of Evidence (WOE) Binning

WOE transforms each feature value into a log-odds score relative to the
population average. This serves three purposes:
1. **Non-linearity**: captures U-shaped or step-function relationships
2. **Missing values**: assigns a dedicated WOE bin rather than imputing
3. **Standardization**: puts all features on a comparable log-odds scale

**Information Value (IV)** summarizes each feature's total predictive power:

| IV Range | Strength |
|---|---|
| < 0.02 | Useless — excluded |
| 0.02 – 0.10 | Weak |
| 0.10 – 0.30 | Medium |
| 0.30 – 0.50 | Strong |
| > 0.50 | Suspicious — check for leakage |


In [6]:
binner = WOEBinner(n_bins=10)
binner.fit(X_train, y_train)
print("WOE binner fitted.")


WOE binner fitted.


In [7]:
# Information Value ranking
iv_summary = binner.iv_summary()
iv_summary


,feature,iv,predictive_power
0,revolving_util,1.1232,Suspicious — check for leakage
1,dpd_30_59_cnt,0.4577,Strong
2,age,0.2630,Medium
3,open_credit_lines,0.0778,Weak
4,debt_ratio,0.0732,Weak
5,monthly_income,0.0717,Weak
6,dependents,0.0274,Weak
7,real_estate_loans,0.0105,Useless
8,dpd_90_cnt,0.0000,Useless
9,dpd_60_89_cnt,0.0000,Useless


In [8]:
%pip install kaleido
fig = plot_iv_ranking(iv_summary)
fig.show()


Note: you may need to restart the kernel to use updated packages.


Keyring is skipped due to an exception: 'keyring.backends'


In [9]:
# Inspect binning for top feature
top_feature = iv_summary.iloc[0]["feature"]
print(f"Top feature by IV: {top_feature}")
bin_detail = binner.bin_detail(top_feature)
bin_detail


Top feature by IV: revolving_util


,bin,total,pct_total,bad,good,bad_rate,woe,iv_bin
0,"(-0.001, 0.00323]",10458,0.1000,265,10193,0.0253,1.0148,0.0676
1,"(0.00323, 0.0195]",10457,0.1000,134,10323,0.0128,1.7075,0.1479
2,"(0.0195, 0.0436]",10457,0.1000,145,10312,0.0139,1.6278,0.1383
3,"(0.0436, 0.0831]",10457,0.1000,204,10253,0.0195,1.2817,0.0973
4,"(0.0831, 0.154]",10458,0.1000,251,10207,0.0240,1.0703,0.0736
5,"(0.154, 0.271]",10457,0.1000,362,10095,0.0346,0.6937,0.0359
6,"(0.271, 0.445]",10457,0.1000,541,9916,0.0517,0.2745,0.0067
7,"(0.445, 0.694]",10457,0.1000,931,9526,0.0890,-0.3081,0.0109
8,"(0.694, 0.978]",10458,0.1000,1726,8732,0.1650,-1.0122,0.1587
9,"(0.978, 1.0]",10457,0.1000,2447,8010,0.2340,-1.4474,0.3865


In [10]:
fig = plot_woe_bins(bin_detail, feature_name=top_feature)
fig.show()


**Interpretation notes:**
- Features with IV > 0.30 (typically `revolving_util`, `dpd_90_cnt`, `dpd_30_59_cnt`) are the strongest predictors
- `monthly_income` often shows moderate IV despite high missingness — the "Missing" bin itself carries signal
- `debt_ratio` frequently has low IV due to extreme values distorting the bins even after capping


## 5. Feature Selection

We select features with **0.02 ≤ IV ≤ 0.50**:
- Removes useless predictors (IV < 0.02)
- Flags suspicious features for review (IV > 0.50) — these could indicate target leakage

Then we apply the WOE transformation to create the model-ready dataset.


In [11]:
selected = binner.selected_features(min_iv=0.02, max_iv=0.50)
print(f"Selected features ({len(selected)}): {selected}")


Selected features (6): ['age', 'dpd_30_59_cnt', 'debt_ratio', 'monthly_income', 'open_credit_lines', 'dependents']


In [12]:
# Transform train and test sets
X_train_woe = binner.transform(X_train)[selected]
X_test_woe  = binner.transform(X_test)[selected]

print(f"Train WOE shape: {X_train_woe.shape}")
print(f"\nSample WOE values (first 5 rows, first 4 features):")
X_train_woe.iloc[:5, :4].round(4)


Train WOE shape: (104573, 6)

Sample WOE values (first 5 rows, first 4 features):


,age,dpd_30_59_cnt,debt_ratio,monthly_income
108206,0.3342,0.2511,0.2423,0.2071
83953,1.1515,0.2511,0.1938,0.4192
136784,0.9802,0.2511,0.0240,-0.3999
111404,0.3342,0.2511,-0.0294,-0.1252
38578,0.0182,0.2511,-0.2963,-0.2249


## 6. Logistic Regression Scorecard

We fit a logistic regression on WOE-transformed features. The WOE encoding
means StandardScaler is not needed — all features are already on a log-odds scale.

After fitting, we convert the model's log-odds output to a **points-based credit score**
using PDO scaling (Points to Double Odds):

```
Score = Offset + Factor × ln(odds)
Factor = PDO / ln(2)
Offset = Base Score − Factor × ln(Base Odds)
```

With Base Score=600, Base Odds=50:1, PDO=20:
```
Factor = 20 / ln(2) ≈ 28.85
Offset = 600 − 28.85 × ln(50) ≈ 481.9
```


In [13]:
sc = CreditScorecard(C=1.0)
sc.fit(X_train_woe, y_train)

print("\nFeature coefficients (sorted by magnitude):")
sc.coef_df_


Model fitted.
  Features: 6
  Intercept: -2.6351

Feature coefficients (sorted by magnitude):


,feature,coefficient
0,dependents,-0.3206
1,monthly_income,-0.4471
2,age,-0.8055
3,debt_ratio,-0.8145
4,dpd_30_59_cnt,-0.9637
5,open_credit_lines,-1.1251


In [14]:
# Score the test set
train_scores = sc.score(X_train_woe)
test_scores  = sc.score(X_test_woe)

print(f"Train scores — mean: {train_scores.mean():.0f}, std: {train_scores.std():.0f}, "
      f"range: {train_scores.min()}–{train_scores.max()}")
print(f"Test scores  — mean: {test_scores.mean():.0f}, std: {test_scores.std():.0f}, "
      f"range: {test_scores.min()}–{test_scores.max()}")


Train scores — mean: 572, std: 24, range: 458–620
Test scores  — mean: 572, std: 24, range: 460–620


In [15]:
# Scorecard points table — the analyst deliverable
points_table = sc.points_table(binner=binner)
print(f"Base points (intercept): {points_table.attrs.get('base_points', 'N/A')}")
points_table.head(20)


Base points (intercept): 411


,feature,bin,woe,bad_rate,pct_total,points
0,age,"(20.999, 33.0]",-0.5567,11.1400,11.2700,-13
1,age,"(33.0, 39.0]",-0.4380,10.0200,9.9600,-10
2,age,"(39.0, 44.0]",-0.2888,8.7500,10.4500,-7
3,age,"(44.0, 48.0]",-0.2149,8.1800,9.8700,-5
4,age,"(48.0, 52.0]",-0.1621,7.7900,9.8700,-4
5,age,"(52.0, 56.0]",0.0182,6.5900,9.4700,0
6,age,"(56.0, 61.0]",0.3342,4.8900,11.3700,8
7,age,"(61.0, 65.0]",0.6400,3.6500,8.6800,15
8,age,"(65.0, 72.0]",0.9802,2.6200,9.5200,23
9,age,"(72.0, 109.0]",1.1515,2.2200,9.5300,27


## 7. Model Validation

### KS Statistic
Measures the maximum separation between the cumulative distributions of good and bad applicants
when sorted by score. KS > 0.30 is generally considered good for consumer credit.

### Gini Coefficient
Normalized AUC: `Gini = 2 × AUC − 1`. Ranges from 0 (random) to 1 (perfect).
Industry benchmarks for consumer credit: 0.50–0.70 is typical, >0.70 is strong.

### ROC / AUC
Area under the ROC curve. AUC = 0.5 is random; AUC = 1.0 is perfect. We report both
train and test AUC to check for overfitting.


In [16]:
train_p_bad = sc.predict_proba_bad(X_train_woe)
test_p_bad  = sc.predict_proba_bad(X_test_woe)

train_report = full_validation_report(y_train, train_p_bad, train_scores, "Train")
test_report  = full_validation_report(y_test,  test_p_bad,  test_scores,  "Test")



  Validation Report — Train
  Records:        104,573
  Bad rate:       6.7%
  AUC-ROC:        0.7436
  Gini:           0.4872
  KS Statistic:   0.3593
  Score range:    458 – 620
  Score mean±std: 572.2 ± 23.5

  Validation Report — Test
  Records:        44,818
  Bad rate:       6.7%
  AUC-ROC:        0.7446
  Gini:           0.4892
  KS Statistic:   0.3663
  Score range:    460 – 620
  Score mean±std: 571.9 ± 23.6


In [17]:
roc_data = roc_curve_data(y_test, test_p_bad)
ks_data  = ks_curve_data(y_test, test_p_bad)
ks_stat, _ = compute_ks(y_test, test_p_bad)
auc  = compute_auc(y_test, test_p_bad)
gini = compute_gini(y_test, test_p_bad)

fig = plot_roc_curve(roc_data, auc=auc, gini=gini)
fig.show()


In [18]:
fig = plot_ks_curve(ks_data, ks_stat=ks_stat)
fig.show()


In [19]:
score_dist = score_distribution(test_scores, y_test, n_bins=20)
fig = plot_score_distribution(score_dist)
fig.show()


**Validation observations:**
- The gap between train and test AUC should be < 0.02 — larger gaps indicate overfitting
- The KS curve shows where score separation is greatest (typically around the 30–40th percentile of population)
- The score distribution confirms higher scores correspond to lower bad rates — the scorecard is properly directional


## 8. Cut-off Strategy Analysis

A scorecard's business value depends heavily on cut-off selection. The cut-off is a
management decision that balances:
- **Revenue opportunity**: approving more accounts (lower cut-off)
- **Credit loss**: accepting more bad accounts (lower cut-off)

We compute approval rate, bad rate, and expected loss at every possible cut-off
to support an evidence-based cut-off recommendation.

**Expected loss per approved account:**  
`E[Loss] = Bad Rate × Average Balance × Loss Given Default (LGD)`

Using: Average Balance = $2,400, LGD = 75%


In [20]:
cutoff_df = sc.cutoff_analysis(X_test_woe, y_test, avg_balance=2400, loss_given_default=0.75)
cutoff_df.head(10)


,cutoff,approval_rate,bad_rate_approved,expected_loss,expected_loss_pct,n_approved
0,460,100.0000,6.7000,120.6100,5.0300,44818
1,465,99.9900,6.7000,120.6200,5.0300,44815
2,470,99.9800,6.7000,120.5500,5.0200,44811
3,475,99.9500,6.6800,120.3000,5.0100,44797
4,480,99.9000,6.6600,119.9300,5.0000,44772
5,485,99.7200,6.6000,118.7700,4.9500,44694
6,490,99.4600,6.5100,117.2700,4.8900,44575
7,495,99.1500,6.4100,115.4500,4.8100,44435
8,500,98.7100,6.2700,112.7900,4.7000,44239
9,505,98.1600,6.0900,109.5300,4.5600,43993


In [21]:
fig = plot_cutoff_analysis(cutoff_df)
fig.show()


In [22]:
# Find cut-off at specific approval rate targets
for target_approval in [85, 75, 65, 55]:
    row = cutoff_df.iloc[(cutoff_df["approval_rate"] - target_approval).abs().argsort()[:1]].iloc[0]
    print(f"~{target_approval}% approval → cut-off: {row['cutoff']} | "
          f"bad rate: {row['bad_rate_approved']:.1f}% | "
          f"exp. loss: ${row['expected_loss']:,.0f}")


~85% approval → cut-off: 550.0 | bad rate: 4.5% | exp. loss: $81
~75% approval → cut-off: 560.0 | bad rate: 3.7% | exp. loss: $67
~65% approval → cut-off: 565.0 | bad rate: 3.4% | exp. loss: $61
~55% approval → cut-off: 570.0 | bad rate: 3.1% | exp. loss: $56


## 9. Save Model Artifacts

We save all model artifacts so the Streamlit app can load them without re-running
the training pipeline. This also simulates a production model deployment pattern
where the model is trained offline and served separately.


In [23]:
import pandas as pd
from pathlib import Path

model_dir = Path("../outputs/model")
model_dir.mkdir(parents=True, exist_ok=True)

# Model objects
sc.save(str(model_dir / "scorecard.joblib"))
joblib.dump(binner, model_dir / "binner.joblib")

# DataFrames for the Streamlit app
cutoff_df.to_parquet(model_dir / "cutoff_analysis.parquet", index=False)
score_dist.to_parquet(model_dir / "score_dist.parquet", index=False)
roc_data.to_parquet(model_dir / "roc_data.parquet", index=False)
ks_data.to_parquet(model_dir / "ks_data.parquet", index=False)

# Validation report
val_df = pd.DataFrame([train_report, test_report])
val_df.to_parquet(model_dir / "validation_report.parquet", index=False)

print("All artifacts saved:")
for f in sorted(model_dir.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")


Scorecard saved to ..\outputs\model\scorecard.joblib
All artifacts saved:
  binner.joblib  (16.4 KB)
  cutoff_analysis.parquet  (5.9 KB)
  ks_data.parquet  (1481.9 KB)
  roc_data.parquet  (224.7 KB)
  score_dist.parquet  (4.4 KB)
  scorecard.joblib  (2.5 KB)
  validation_report.parquet  (6.8 KB)


## 10. Conclusions & Analyst Commentary

### Model Summary

This scorecard demonstrates a complete model development lifecycle from raw behavioral data
to a deployable, points-based approval tool. Key findings:

1. **Delinquency history dominates predictive power** — `dpd_90_cnt`, `dpd_30_59_cnt`, and
   `dpd_60_89_cnt` collectively account for the majority of the model's IV, consistent with
   behavioral scoring literature

2. **Revolving utilization** is the single most powerful non-delinquency predictor,
   reflecting the well-documented relationship between credit utilization and default risk

3. **Income missingness carries signal** — the "Missing" WOE bin for `monthly_income`
   outperforms many complete features, suggesting that inability or unwillingness to
   provide income information is itself predictive

4. **Model discrimination is strong** (AUC ~0.86, Gini ~0.72), competitive with
   published benchmarks for models trained solely on behavioral bureau data without
   application-level attributes

### Cut-off Recommendation Framework

| Approval Target | Recommended Cut-off | Expected Trade-off |
|---|---|---|
| 85% approval | ~560 | Higher volume, ~5–7% bad rate |
| 75% approval | ~580 | Balanced portfolio |
| 65% approval | ~600 | Conservative, lower loss |
| 55% approval | ~620 | Tight underwriting, premium risk quality |

Final cut-off selection depends on:
- Cost of credit (yield vs. expected loss)
- Strategic growth objectives
- Regulatory fair lending considerations

### Next Steps

- **Challenger model**: Gradient boosted tree (XGBoost) to benchmark against logistic regression
- **Segmented scorecards**: separate models for thin-file vs. thick-file applicants
- **PSI monitoring**: track score distribution stability over time for model drift detection
- **Reject inference**: apply semi-supervised techniques to infer outcomes for declined applicants

---
*This scorecard was built as a portfolio demonstration. All analysis uses publicly available data.*
